<a href="https://colab.research.google.com/github/tiagosoaresmm/projeto-dashboard-b2b/blob/main/pedidos_b2b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup caminhos dos **arquivos**

In [33]:
import pandas as pd
from datetime import datetime
from google.colab import drive
from google.colab import auth
from google.auth import default
import gspread
import os # Importar o módulo os para listar arquivos

# ==========================================
# 1. AUTENTICAÇÃO E CONEXÃO (MODO MVP)
# ==========================================
print("🔄 Passo 1: Solicitando acesso ao seu Drive e Sheets...")
drive.mount('/content/drive')
auth.authenticate_user()

creds, _ = default()
gc = gspread.authorize(creds)
print("✅ Autenticação concluída com sucesso!\n")

🔄 Passo 1: Solicitando acesso ao seu Drive e Sheets...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Autenticação concluída com sucesso!



In [34]:
# ==========================================
# 2. CONFIGURAÇÃO DOS CAMINHOS DOS ARQUIVOS
# ==========================================
# Seus caminhos exatos salvos em variáveis de texto (strings)
# Base directory where the CSV files are located
base_dir = '/content/drive/Shareddrives/MM - Consultoria Interna/#04_B2B/Automação dados do pedido/Projeto_Dashboard_B2B/'

# Prefixes for the sales and supply files
prefix_vendas = 'cubo_de_vendas_red sales_red'
prefix_supply = 'pedido_supply pedidos_supply'
prefix_clientes = 'clientes clientes'

# Function to find the latest file matching a prefix
def find_latest_csv(directory, prefix):
    matching_files = []
    for filename in os.listdir(directory):
        if filename.startswith(prefix) and filename.endswith('.csv'):
            matching_files.append(os.path.join(directory, filename))

    if not matching_files:
        return None

    # Sort files by modification time (most recent first)
    matching_files.sort(key=os.path.getmtime, reverse=True)
    return matching_files[0]

# Dynamically find the paths
path_vendas = find_latest_csv(base_dir, prefix_vendas)
path_supply = find_latest_csv(base_dir, prefix_supply)
path_clientes = find_latest_csv(base_dir, prefix_clientes)

if not path_vendas:
    raise FileNotFoundError(f"No sales CSV file found starting with '{prefix_vendas}' in '{base_dir}'")
if not path_supply:
    raise FileNotFoundError(f"No supply CSV file found starting with '{prefix_supply}' in '{base_dir}'")
if not path_clientes:
    raise FileNotFoundError(f"Nenhum arquivo de clientes encontrado com '{prefix_clientes}' em '{base_dir}'")

print(f"📦 Arquivo de vendas encontrado: {path_vendas}")
print(f"📦 Arquivo de supply encontrado: {path_supply}")
print(f"📦 Arquivo de clientes encontrado: {path_clientes}")

# Nome exato da Planilha Google que você criou para ser o seu banco de dados do Looker
NOME_DA_PLANILHA_DESTINO = "Base_Consolidada_Pedidos"

📦 Arquivo de vendas encontrado: /content/drive/Shareddrives/MM - Consultoria Interna/#04_B2B/Automação dados do pedido/Projeto_Dashboard_B2B/cubo_de_vendas_red sales_red 2026-05-26T1232.csv
📦 Arquivo de supply encontrado: /content/drive/Shareddrives/MM - Consultoria Interna/#04_B2B/Automação dados do pedido/Projeto_Dashboard_B2B/pedido_supply pedidos_supply 2026-05-26T1228.csv
📦 Arquivo de clientes encontrado: /content/drive/Shareddrives/MM - Consultoria Interna/#04_B2B/Automação dados do pedido/Projeto_Dashboard_B2B/clientes clientes 2026-05-26T1244.csv


In [35]:
# ==========================================
# 3. LEITURA E HIGIENIZAÇÃO DOS DADOS (ETL)
# ==========================================
print("🔄 Passo 2: Carregando os arquivos CSV e forçando tipagem...")

# Dicionários de tipagem para proteger os IDs e Flags (Evita perda de zero e '.0')
tipos_vendas = {
    'Vendas ID Pedido': str,
    'Vendas ID Pedido Pai': str,
    'Vendas ID Cliente': str
}

tipos_supply = {
    'Pedidos Supply Cpf/Cnpj Cliente': str,
    'Pedidos Supply ID Item Pedido Venda': str,
    'Pedidos Supply Pedido Venda': str,
    'Pedidos Supply Pedido Venda Original': str,
    'Pedidos Supply Aging Chão': str,
    'Pedidos Supply Flag Perdas': str,
    'Pedidos Supply Flag Cancelado': str,
    'Pedidos Supply Flag Pedido Venda Cancelado': str,
    'Pedidos Supply Flag OC Cancelada': str,
    'Pedidos Supply Flag NF Madeira Cancelada': str
}

tipos_clientes = {
    'Clientes CPF / CNPJ': str,
    'Clientes ID do Cliente': str
}

# O parâmetro dtype força o pandas a respeitar os tipos mapeados acima
df_vendas = pd.read_csv(path_vendas, dtype=tipos_vendas, low_memory=False)
df_supply = pd.read_csv(path_supply, dtype=tipos_supply, low_memory=False)
df_clientes = pd.read_csv(path_clientes, dtype=tipos_clientes, low_memory=False)

print("🔄 Passo 3: Limpando as chaves de cruzamento (Regex)...")
# Garante que o ID de vendas seja string (redundância segura)
df_vendas['vendas_id_str'] = df_vendas['Vendas ID Pedido'].astype(str)
# Chave de Vendas para o Join com Clientes
df_vendas['vendas_id_cliente_str'] = df_vendas['Vendas ID Cliente'].astype(str).str.replace(r'\.0$', '', regex=True)


# Remove letras e prefixos do ID de Supply (ex: 'Z64985914' vira '64985914') mantendo como texto puro
df_supply['pedido_venda_limpo'] = df_supply['Pedidos Supply Pedido Venda'].astype(str).str.replace(r'\D', '', regex=True)

print("🔄 Limpando as chaves de cruzamento de Clientes...")
# Remove decimais '.0' indesejados caso o pandas interprete como float antes da conversão
df_clientes['Clientes ID do Cliente'] = df_clientes['Clientes ID do Cliente'].astype(str).str.replace(r'\.0$', '', regex=True)
# Protege o CPF/CNPJ com apóstrofo
df_clientes['Clientes CPF / CNPJ'] = "'" + df_clientes['Clientes CPF / CNPJ'].astype(str)

# ==========================================
# TRATAMENTO DE FORMATOS (CPF/CNPJ E PRAZOS)
# ==========================================
print("🔄 Tratando casas decimais, '.0' nos prazos e protegendo CPF/CNPJ...")

# Protege o CPF/CNPJ adicionando um apóstrofo oculto para o Excel/Sheets não cortar os zeros
df_supply['Pedidos Supply Cpf/Cnpj Cliente'] = "'" + df_supply['Pedidos Supply Cpf/Cnpj Cliente'].astype(str)

colunas_prazo = [
    'Pedidos Supply Prazo Prometido ao Cliente (Total)',
    'Pedidos Supply Prazo Prometido Fornecedor',
    'Pedidos Supply Prazo Realizado Fornecedor'
]

for col in colunas_prazo:
    # Verifica se a coluna existe no dataframe atual (para evitar erro caso use no df de vendas ou supply)
    if col in df_supply.columns:
        df_supply[col] = (
            df_supply[col]
            .astype(str)                          # 1. Converte tudo para texto
            .str.replace(r'\.0$', '', regex=True) # 2. Remove o ".0" exato do final da string
            .str.replace('.', ',', regex=False)   # 3. Troca qualquer ponto restante (decimal) por vírgula
            .replace('nan', '')                   # 4. Troca os textos 'nan' (vazios originais) por string vazia
        )

🔄 Passo 2: Carregando os arquivos CSV e forçando tipagem...
🔄 Passo 3: Limpando as chaves de cruzamento (Regex)...
🔄 Limpando as chaves de cruzamento de Clientes...
🔄 Tratando casas decimais, '.0' nos prazos e protegendo CPF/CNPJ...


# **Cruzamento das bases**

In [36]:
import pandas as pd
import numpy as np
from datetime import datetime

# ==========================================
# 4. CRUZAMENTO DAS BASES (JOIN)
# ==========================================
print("🔄 Passo 4: Realizando o Join entre Vendas, Supply e Clientes...")

# 1. Primeiro Join (Vendas + Supply)
df_consolidado = pd.merge(
    df_vendas,
    df_supply,
    left_on='vendas_id_str',
    right_on='pedido_venda_limpo',
    how='outer',
    indicator=True
)

df_consolidado['Join feito com sucesso'] = df_consolidado['_merge'].apply(lambda x: 1 if x == 'both' else 0)
df_consolidado = df_consolidado.drop(columns=['_merge'])

# 2. Segundo Join (Consolidado + Clientes)
# Cruzamos o 'vendas_id_cliente_str' do df_consolidado com o 'Clientes ID do Cliente' do df_clientes
df_consolidado = pd.merge(
    df_consolidado,
    df_clientes,
    left_on='vendas_id_cliente_str', # Nova chave vinda do cubo de vendas
    right_on='Clientes ID do Cliente', # Chave da base de clientes
    how='left'
)

# Limpeza para garantir que o CSV final do Join fique limpo
df_consolidado = df_consolidado.replace([np.inf, -np.inf], np.nan)
df_consolidado = df_consolidado.astype(str)
df_consolidado = df_consolidado.replace(['nan', 'NaN', 'NaT', '<NA>', 'None'], '')

# ==========================================
# 4.1 VALIDAÇÃO DO CRUZAMENTO (JOIN)
# ==========================================
print("🔄 Passo 4.0.1: Validando a integridade do Join para pedidos 1P Aprovados...")

# Definindo o nome correto das colunas
col_vendedor = 'TDV ID Vendedor Tdv' # Verifique o case-sensitive na sua base (Tdv ou tdv)
col_tipo_venda = 'Vendas Tipo Venda'
col_situacao = 'Vendas Situacao'

# Verificamos se as colunas existem para evitar erros (KeyError)
if all(col in df_consolidado.columns for col in [col_vendedor, col_tipo_venda, col_situacao]):

    # 1. Aplicando os filtros combinados
    # Usamos != '' porque os valores nulos viraram string vazia no passo anterior
    filtro_validacao = (
        (df_consolidado[col_vendedor] != '') &
        (df_consolidado[col_tipo_venda] == '1P') &
        (df_consolidado[col_situacao] == 'Aprovado')
    )

    df_alvo_validacao = df_consolidado[filtro_validacao]
    total_alvo = len(df_alvo_validacao)

    if total_alvo > 0:
        # 2. Verificando se o 'Join feito com sucesso' falhou ('0') em algum caso
        # Buscamos a string '0' por causa do astype(str) aplicado antes
        df_join_falho = df_alvo_validacao[df_alvo_validacao['Join feito com sucesso'] == '0']
        total_falhas = len(df_join_falho)

        print(f"📊 Foram encontrados {total_alvo} pedidos alvo (1P, Aprovado, com Vendedor).")

        if total_falhas == 0:
            print("✅ VALIDAÇÃO OK: 100% dos pedidos cruzaram com sucesso na base de Supply (Join = 1).")
        else:
            print(f"⚠️ ALERTA DE INCONSISTÊNCIA: {total_falhas} pedido(s) não cruzaram com a tabela de Supply!")

            # (Opcional) Imprime até 5 IDs de vendas que deram problema para facilitar o rastreio no seu CSV
            pedidos_com_falha = df_join_falho['vendas_id_str'].head(5).tolist()
            print(f"   Exemplos de IDs que não estão em Supply: {pedidos_com_falha}")
    else:
        print("ℹ️ Nenhum pedido nesta base atendeu aos 3 critérios simultaneamente.")
else:
    print("❌ ALERTA: Uma ou mais colunas usadas no filtro não existem no DataFrame. Verifique a nomenclatura exata.")

# ==========================================
# 4.2 LIMPEZA DA BASE (FILTRO 1P E VENDEDOR PREENCHIDO)
# ==========================================
print("🔄 Passo 4.0.2: Limpando a base consolidada para manter apenas 1P e Vendedor preenchido...")

# Utilizando as variáveis de coluna já definidas no passo anterior (ou redefinindo se necessário)
col_vendedor = 'TDV ID Vendedor Tdv' # Verifique o case-sensitive exato da sua base
col_tipo_venda = 'Vendas Tipo Venda'

# Verifica se as colunas existem antes de filtrar
if all(col in df_consolidado.columns for col in [col_vendedor, col_tipo_venda]):

    total_antes_limpeza = len(df_consolidado)

    # 1. Aplicando os filtros para a limpeza definitiva
    # Vendedor não é string vazia E Tipo Venda é exatamente '1P'
    filtro_limpeza = (
        (df_consolidado[col_vendedor] != '') &
        (df_consolidado[col_tipo_venda] == '1P')
    )

    # 2. Sobrescrevendo o DataFrame consolidado apenas com as linhas filtradas
    # Usamos .copy() para evitar o aviso SettingWithCopyWarning do Pandas
    df_consolidado = df_consolidado[filtro_limpeza].copy()

    total_depois_limpeza = len(df_consolidado)
    linhas_removidas = total_antes_limpeza - total_depois_limpeza

    print(f"🧹 Limpeza concluída com sucesso!")
    print(f"   - Linhas antes: {total_antes_limpeza}")
    print(f"   - Linhas depois: {total_depois_limpeza}")
    print(f"   - Total de linhas removidas (3P ou sem vendedor): {linhas_removidas}")

else:
    print("❌ ALERTA: As colunas necessárias para a limpeza não foram encontradas. A base não foi filtrada.")

# ==========================================
# 4.3 SALVAR JOIN COMO CSV DIRETO NO DRIVE MONTADO
# ==========================================
data_hora_atual = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
nome_arquivo_csv = f"join_tabelas_{data_hora_atual}.csv"

# Caminho da pasta específica no Drive Compartilhado
caminho_diretorio = '/content/drive/Shareddrives/MM - Consultoria Interna/#04_B2B/Automação dados do pedido/Projeto_Dashboard_B2B/'
caminho_completo_salvamento = f"{caminho_diretorio}{nome_arquivo_csv}"

print(f"🔄 Passo 4.1: Gravando o arquivo diretamente no Drive Compartilhado...")

try:
    # O Pandas grava o arquivo direto na pasta mapeada do Drive
    df_consolidado.to_csv(caminho_completo_salvamento, index=False, encoding='utf-8')

    print(f"✅ SUCESSO! Arquivo salvo diretamente na pasta do projeto.")
    print(f"📁 Caminho: {caminho_completo_salvamento}")

except FileNotFoundError:
    print("❌ Erro: O caminho da pasta não foi encontrado. Certifique-se de que rodou o drive.mount('/content/drive') antes deste passo.")
except Exception as e:
    print(f"❌ Erro ao salvar o arquivo no Drive: {e}")

🔄 Passo 4: Realizando o Join entre Vendas, Supply e Clientes...
🔄 Passo 4.0.1: Validando a integridade do Join para pedidos 1P Aprovados...
📊 Foram encontrados 1252 pedidos alvo (1P, Aprovado, com Vendedor).
⚠️ ALERTA DE INCONSISTÊNCIA: 2 pedido(s) não cruzaram com a tabela de Supply!
   Exemplos de IDs que não estão em Supply: ['65396435', '65396435']
🔄 Passo 4.0.2: Limpando a base consolidada para manter apenas 1P e Vendedor preenchido...
🧹 Limpeza concluída com sucesso!
   - Linhas antes: 888748
   - Linhas depois: 1572
   - Total de linhas removidas (3P ou sem vendedor): 887176
🔄 Passo 4.1: Gravando o arquivo diretamente no Drive Compartilhado...
✅ SUCESSO! Arquivo salvo diretamente na pasta do projeto.
📁 Caminho: /content/drive/Shareddrives/MM - Consultoria Interna/#04_B2B/Automação dados do pedido/Projeto_Dashboard_B2B/join_tabelas_2026-05-26_15-54-46.csv


# Preparação dos dados e **limpeza**

In [37]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
import glob

# ==========================================
# PASSO 1: CARREGAMENTO DINÂMICO E LIMPEZA
# ==========================================

# 1.1. Função para encontrar o arquivo mais recente
def find_latest_csv(directory, prefix):
    # Procura todos os arquivos na pasta que começam com o prefixo e terminam em .csv
    padrao_busca = os.path.join(directory, f"{prefix}*.csv")
    arquivos = glob.glob(padrao_busca)

    if not arquivos:
        raise FileNotFoundError(f"❌ Nenhum arquivo encontrado com o prefixo '{prefix}' no diretório: {directory}")

    # Retorna o arquivo com a data de modificação mais recente
    arquivo_mais_recente = max(arquivos, key=os.path.getmtime)
    return arquivo_mais_recente

# 1.2. Definição dos caminhos
base_dir = '/content/drive/Shareddrives/MM - Consultoria Interna/#04_B2B/Automação dados do pedido/Projeto_Dashboard_B2B/'
prefix_join = 'join_tabelas'

print("🔄 Buscando o arquivo mais recente no Drive...")
caminho_do_arquivo = find_latest_csv(base_dir, prefix_join)
print(f"📄 Arquivo encontrado: {os.path.basename(caminho_do_arquivo)}")

# 1.3. Carregamento da Base
# Adicionamos o dtype=str para que todo o arquivo de Join seja lido como texto
# Isso protege os IDs. O seu código já converte as datas corretamente logo abaixo, então não haverá conflito.
df = pd.read_csv(caminho_do_arquivo, dtype=str, low_memory=False)
print(f"✅ Base carregada com {len(df)} linhas.")

# 1.4. Definir o "Hoje" para cálculo de dias corridos em atrasos
hoje = pd.Timestamp(datetime.now().date())

# 1.5. Limpeza das Datas
colunas_de_data = [
    'Vendas Data Compra Date',
    'Cubo de Vendas Data de Aprovação Date',
    'Pedidos Supply - Datas Data Criação OC Date',
    'Pedidos Supply - Datas Limite Data Limite Criação OC Date',
    'Pedidos Supply - Datas Data Faturamento Date',
    'Pedidos Supply - Datas Data de Disponibilidade Fornecedor Date',
    'Pedidos Supply - Datas Limite Data Limite Fornecedor Date',
    'Erros e Desvios Sistêmicos Data da Ocorrência Date',
    'Pedidos Supply - Datas Data Coleta Date',
    'Pedidos Supply - Datas Limite Data Limite Coleta First Mile Date',
    'Pedidos Supply - Datas Data Saida Origem Redespacho Date',
    'Pedidos Supply - Datas Limite Data Limite Redespacho Date',
    'Pedidos Supply - Datas Data Em Rota de Entrega Last Mile Date',
    'Pedidos Supply - Datas Data Entregue Date',
    'Pedidos Supply - Datas Limite Data Limite Last Mile Date'
]

for col in colunas_de_data:
    if col in df.columns:
        df[col] = df[col].replace('1900-01-01', np.nan)
        df[col] = pd.to_datetime(df[col], errors='coerce')

# 1.6. Limpeza de Flags
colunas_flags = [
    'Pedidos Supply Flag OC Cancelada',
    'Pedidos Supply Flag NF Madeira Cancelada',
    'Pedidos Supply Flag Perdas'
]
for col in colunas_flags:
    if col in df.columns:
        df[col] = df[col].fillna(0).astype(int)

print("✅ Limpeza de datas e flags concluída!")

🔄 Buscando o arquivo mais recente no Drive...
📄 Arquivo encontrado: join_tabelas_2026-05-26_15-54-46.csv
✅ Base carregada com 1572 linhas.
✅ Limpeza de datas e flags concluída!


# **Aplicação das regras de negócio:**

[Documentação Regras de Negócio](https://docs.google.com/document/d/1E35TDjs7ledF2fyDMoVI8OZnE04ugbkeEfpCg5KMe1M/edit?usp=sharing)

In [38]:
# ==========================================
# PASSO 2: REGRAS DOS MARCOS
# ==========================================

# ==========================================
# MARCO 1 - APROVAÇÃO E OC
# ==========================================

print("\n🔄 Processando Marco 1: Aprovação e OC...")

# 2.1 Criando as colunas base do Marco 1
df['M1_Aprovacao_OC_Data'] = df['Pedidos Supply - Datas Data Criação OC Date']

# 2.2 Definindo as condições de forma booleana (Máscaras)
dias_desde_compra = (hoje - df['Vendas Data Compra Date']).dt.days

m1_criada = df['Pedidos Supply - Datas Data Criação OC Date'].notna()
m1_criada_no_prazo = m1_criada & (df['Pedidos Supply - Datas Data Criação OC Date'] <= df['Pedidos Supply - Datas Limite Data Limite Criação OC Date'])
m1_criada_atraso = m1_criada & (df['Pedidos Supply - Datas Data Criação OC Date'] > df['Pedidos Supply - Datas Limite Data Limite Criação OC Date'])

# Regras Críticas (Vermelho)
m1_cancelada = df['Pedidos Supply Flag OC Cancelada'] == 1
m1_sem_aprovacao = df['Cubo de Vendas Data de Aprovação Date'].isna() & (dias_desde_compra > 3)
m1_atraso_criacao = (~m1_criada) & (hoje > df['Pedidos Supply - Datas Limite Data Limite Criação OC Date'])

# 2.3 Aplicando a hierarquia
condicoes_m1 = [m1_cancelada, m1_sem_aprovacao, m1_atraso_criacao, m1_criada_atraso, m1_criada_no_prazo]
farol_m1 = ['🔴', '🔴', '🔴', '🔴', '🟢']
detalhes_m1 = [
    'Cancelado antes da OC',
    'Travado em Aprovação (>3 dias)',
    'Atraso na Criação OC',
    'OC Criada com Atraso',
    'OC Criada no Prazo'
]

# np.select aplica a regra. O "default" é a cor cinza.
df['M1_Aprovacao_OC_Farol'] = np.select(condicoes_m1, farol_m1, default='⚪')
df['M1_Aprovacao_OC_Detalhe'] = np.select(condicoes_m1, detalhes_m1, default='Em análise/Andamento')

print("✅ Marco 1 concluído!")

# ----------------------------------------------------
# MARCO 2: OPERAÇÃO FORNECEDOR
# ----------------------------------------------------
print("🔄 Processando Marco 2: Fornecedor...")

# A data realizada do Marco 2 será a maior data entre o Bip e o Faturamento
df['M2_Fornecedor_Data'] = df[['Pedidos Supply - Datas Data Faturamento Date',
                               'Pedidos Supply - Datas Data de Disponibilidade Fornecedor Date']].max(axis=1)

# Condições Booleanas
m2_erro_sistemico = df['Erros e Desvios Sistêmicos Data da Ocorrência Date'].notna()
m2_nf_cancelada = df['Pedidos Supply Flag NF Madeira Cancelada'] == 1

m2_bip_ok = df['Pedidos Supply - Datas Data de Disponibilidade Fornecedor Date'].notna()
m2_fat_ok = df['Pedidos Supply - Datas Data Faturamento Date'].notna()
m2_tudo_concluido = m2_bip_ok & m2_fat_ok

m2_atraso_operacao = (~m2_tudo_concluido) & (hoje > df['Pedidos Supply - Datas Limite Data Limite Fornecedor Date'])

m2_concluido_no_prazo = m2_tudo_concluido & (df['M2_Fornecedor_Data'] <= df['Pedidos Supply - Datas Limite Data Limite Fornecedor Date'])
m2_concluido_com_atraso = m2_tudo_concluido & (df['M2_Fornecedor_Data'] > df['Pedidos Supply - Datas Limite Data Limite Fornecedor Date'])

# Hierarquia Marco 2
condicoes_m2 = [m2_erro_sistemico, m2_nf_cancelada, m2_atraso_operacao, m2_concluido_com_atraso, m2_concluido_no_prazo]
farol_m2 = ['🔴', '🔴', '🔴', '🔴', '🟢']
detalhes_m2 = [
    'Erro Sistêmico Fornecedor',
    'NF Cancelada Origem',
    'Atraso Operação Fornecedor',
    'Operação Concluída com Atraso',
    'Liberado no Prazo'
]

df['M2_Fornecedor_Farol'] = np.select(condicoes_m2, farol_m2, default='⚪')
df['M2_Fornecedor_Detalhe'] = np.select(condicoes_m2, detalhes_m2, default='Aguardando Operação Fornecedor')


# ----------------------------------------------------
# MARCO 3: FIRST MILE (COLETA)
# ----------------------------------------------------
print("🔄 Processando Marco 3: First Mile...")

df['M3_Coleta_Data'] = df['Pedidos Supply - Datas Data Coleta Date']

m3_perdas = df['Pedidos Supply Flag Perdas'] == 1
m3_coleta_ok = df['Pedidos Supply - Datas Data Coleta Date'].notna()

m3_atraso_coleta = (~m3_coleta_ok) & (hoje > df['Pedidos Supply - Datas Limite Data Limite Coleta First Mile Date'])
m3_coletado_no_prazo = m3_coleta_ok & (df['Pedidos Supply - Datas Data Coleta Date'] <= df['Pedidos Supply - Datas Limite Data Limite Coleta First Mile Date'])
m3_coletado_com_atraso = m3_coleta_ok & (df['Pedidos Supply - Datas Data Coleta Date'] > df['Pedidos Supply - Datas Limite Data Limite Coleta First Mile Date'])

# Hierarquia Marco 3
condicoes_m3 = [m3_perdas, m3_atraso_coleta, m3_coletado_com_atraso, m3_coletado_no_prazo]
farol_m3 = ['🔴', '🔴', '🔴', '🟢']
detalhes_m3 = [
    'Extravio/Avaria Identificada',
    'Atraso na Coleta',
    'Coletado com Atraso',
    'Coletado no Prazo'
]

df['M3_Coleta_Farol'] = np.select(condicoes_m3, farol_m3, default='⚪')
df['M3_Coleta_Detalhe'] = np.select(condicoes_m3, detalhes_m3, default='Aguardando Coleta')


# ----------------------------------------------------
# MARCO 4: MIDDLE MILE (REDESPACHO)
# ----------------------------------------------------
print("🔄 Processando Marco 4: Middle Mile...")

df['M4_Redespacho_Data'] = df['Pedidos Supply - Datas Data Saida Origem Redespacho Date']

m4_perdas = df['Pedidos Supply Flag Perdas'] == 1
m4_saida_ok = df['Pedidos Supply - Datas Data Saida Origem Redespacho Date'].notna()

m4_atraso_redespacho = (~m4_saida_ok) & (hoje > df['Pedidos Supply - Datas Limite Data Limite Redespacho Date'])
m4_redespacho_no_prazo = m4_saida_ok & (df['Pedidos Supply - Datas Data Saida Origem Redespacho Date'] <= df['Pedidos Supply - Datas Limite Data Limite Redespacho Date'])
m4_redespacho_com_atraso = m4_saida_ok & (df['Pedidos Supply - Datas Data Saida Origem Redespacho Date'] > df['Pedidos Supply - Datas Limite Data Limite Redespacho Date'])

# Hierarquia Marco 4
condicoes_m4 = [m4_perdas, m4_atraso_redespacho, m4_redespacho_com_atraso, m4_redespacho_no_prazo]
farol_m4 = ['🔴', '🔴', '🔴', '🟢']
detalhes_m4 = [
    'Extravio/Avaria Identificada',
    'Atraso no Redespacho',
    'Redespachado com Atraso',
    'Redespachado no Prazo'
]

df['M4_Redespacho_Farol'] = np.select(condicoes_m4, farol_m4, default='⚪')
df['M4_Redespacho_Detalhe'] = np.select(condicoes_m4, detalhes_m4, default='Aguardando Redespacho')


# ----------------------------------------------------
# MARCO 5: LAST MILE (ENTREGA FINAL)
# ----------------------------------------------------
print("🔄 Processando Marco 5: Last Mile...")

df['M5_Entrega_Data'] = df['Pedidos Supply - Datas Data Entregue Date']

m5_perdas = df['Pedidos Supply Flag Perdas'] == 1
m5_entregue_ok = df['Pedidos Supply - Datas Data Entregue Date'].notna()

# Verifica o atraso pela data limite de Last Mile que identificamos na base
m5_atraso_entrega = (~m5_entregue_ok) & (hoje > df['Pedidos Supply - Datas Limite Data Limite Last Mile Date'])
m5_entregue_no_prazo = m5_entregue_ok & (df['Pedidos Supply - Datas Data Entregue Date'] <= df['Pedidos Supply - Datas Limite Data Limite Last Mile Date'])
m5_entregue_com_atraso = m5_entregue_ok & (df['Pedidos Supply - Datas Data Entregue Date'] > df['Pedidos Supply - Datas Limite Data Limite Last Mile Date'])

# Hierarquia Marco 5
condicoes_m5 = [m5_perdas, m5_atraso_entrega, m5_entregue_com_atraso, m5_entregue_no_prazo]
farol_m5 = ['🔴', '🔴', '🔴', '🟢']
detalhes_m5 = [
    'Extravio/Avaria em Rota',
    'Atraso na Entrega Final',
    'Entregue com Atraso',
    'Entregue no Prazo'
]

df['M5_Entrega_Farol'] = np.select(condicoes_m5, farol_m5, default='⚪')
df['M5_Entrega_Detalhe'] = np.select(condicoes_m5, detalhes_m5, default='Aguardando Entrega')

print("✅ Todos os Marcos processados com sucesso!")



🔄 Processando Marco 1: Aprovação e OC...
✅ Marco 1 concluído!
🔄 Processando Marco 2: Fornecedor...
🔄 Processando Marco 3: First Mile...
🔄 Processando Marco 4: Middle Mile...
🔄 Processando Marco 5: Last Mile...
✅ Todos os Marcos processados com sucesso!


In [41]:
# ==========================================
# PASSO 4: STATUS GERAL, RECORTE E SALVAMENTO SEGURO
# ==========================================
print("\n🔄 Iniciando etapa final de consolidação e salvamento...")

# 4.1. Regra para o Status Geral do Pedido (Refinada)

# Prioridade 1: O pedido foi entregue no prazo? (Isso perdoa os vermelhos anteriores)
entregue_no_prazo = (df['M5_Entrega_Farol'] == '🟢')

# Prioridade 2: Existe algum vermelho em qualquer marco do ciclo?
tem_vermelho = (
    (df['M1_Aprovacao_OC_Farol'] == '🔴') |
    (df['M2_Fornecedor_Farol'] == '🔴') |
    (df['M3_Coleta_Farol'] == '🔴') |
    (df['M4_Redespacho_Farol'] == '🔴') |
    (df['M5_Entrega_Farol'] == '🔴')
)

# Prioridade 3: O pedido foi concluído, porém com atraso (Garante que fique verde no fim da vida útil do pedido, mas o M5 indicará o atraso)
entregue_com_atraso = (df['M5_Entrega_Detalhe'] == 'Entregue com Atraso')

# A ORDEM AQUI É CRÍTICA! O np.select avalia da primeira para a última condição.
# Como 'entregue_no_prazo' é o primeiro item da lista, se ele for verdadeiro (True),
# o Pandas vai aplicar '🟢 Concluído' e ignorar completamente o fato de existir a flag 'tem_vermelho'.
condicoes_gerais = [entregue_no_prazo, tem_vermelho, entregue_com_atraso]
resultados_gerais = ['🟢 Concluído', '🔴 Requer Atenção', '🟢 Concluído']

df['Status_Geral_Pedido'] = np.select(condicoes_gerais, resultados_gerais, default='⚪ Em Andamento Normal')

# 4.2. Definição estrita das colunas que queremos levar (na ordem correta)
colunas_exportacao = [
    'TDV ID Vendedor Tdv',
    'Vendas Tipo Pessoa',
    'Vendas ID Pedido',
    'Vendas ID Pedido Pai',
    'Pedidos Supply ID Item Pedido Venda',
    'Clientes ID do Cliente',
    'Clientes CPF / CNPJ',
    'Clientes Nome',
    'Produto Nome Produto',
    'Produto Categoria Categoria Nova estrutura',
    'Dimensão Produto Nome Fornecedor',
    'Fornecedores UF Fornecedor',
    'Vendas Data Compra Date',
    'Pedidos Supply Prazo Prometido ao Cliente (Total)',
    'Status_Geral_Pedido',
    'M1_Aprovacao_OC_Data', 'M1_Aprovacao_OC_Farol', 'M1_Aprovacao_OC_Detalhe',
    'M2_Fornecedor_Data', 'M2_Fornecedor_Farol', 'M2_Fornecedor_Detalhe',
    'M3_Coleta_Data', 'M3_Coleta_Farol', 'M3_Coleta_Detalhe',
    'M4_Redespacho_Data', 'M4_Redespacho_Farol', 'M4_Redespacho_Detalhe',
    'M5_Entrega_Data', 'M5_Entrega_Farol', 'M5_Entrega_Detalhe'
]

# Criar o dataframe final filtrando apenas as colunas desejadas
df_exportacao = df[colunas_exportacao].copy()

# Tratamento para as datas virarem strings limpas antes de salvar
for col in df_exportacao.columns:
    if pd.api.types.is_datetime64_any_dtype(df_exportacao[col]):
        df_exportacao[col] = df_exportacao[col].dt.strftime('%Y-%m-%d')

df_exportacao = df_exportacao.fillna('')

# 4.3. Criação Dinâmica do Nome do Arquivo com a Data de Hoje
data_hoje_str = hoje.strftime('%Y-%m-%d') # Gera "2026-05-25" (ou a data da execução)
nome_arquivo_novo = f"pedidos_dash_{data_hoje_str}.csv"

# Caminho mandatório definido por você
pasta_destino = '/content/drive/Shareddrives/MM - Consultoria Interna/#04_B2B/Automação dados do pedido/Projeto_Dashboard_B2B/'
caminho_final_salvamento = os.path.join(pasta_destino, nome_arquivo_novo)

print(f"🔄 Gerando novo arquivo: {nome_arquivo_novo}...")
print(f"📁 Salvando na pasta mandatória do Drive...")

try:
    # Garante que a pasta destino existe (proteção extra do código)
    if not os.path.exists(pasta_destino):
        os.makedirs(pasta_destino)

    # Salva o DataFrame como um arquivo CSV novo diretamente na pasta compartilhada do Drive
    # Usamos o encoding 'utf-8-sig' para que acentos e emojis de farol funcionem perfeitamente no Excel/Looker
    df_exportacao.to_csv(caminho_final_salvamento, index=False, encoding='utf-8-sig')

    print("\n🚀 ARQUIVO HISTÓRICO GERADO COM SUCESSO!")
    print(f"📍 Salvo em: {caminho_final_salvamento}")

except Exception as e:
    print(f"❌ Erro ao salvar o arquivo físico no Drive: {e}")
    print("Verifique se o caminho da pasta compartilhada está correto ou se o Drive está montado.")


🔄 Iniciando etapa final de consolidação e salvamento...
🔄 Gerando novo arquivo: pedidos_dash_2026-05-26.csv...
📁 Salvando na pasta mandatória do Drive...

🚀 ARQUIVO HISTÓRICO GERADO COM SUCESSO!
📍 Salvo em: /content/drive/Shareddrives/MM - Consultoria Interna/#04_B2B/Automação dados do pedido/Projeto_Dashboard_B2B/pedidos_dash_2026-05-26.csv
